In [46]:
# Import Libraries

import pandas as pd
import numpy as np
import torch
import re

from sklearn.model_selection import train_test_split

from transformers import AutoTokenizer  # For Tokenization for BERT

from torch.utils.data import Dataset    # To patch all the encoding in single datasets
from torch.utils.data import DataLoader # Sends samples to BERT in pack of chunks to fasten up traning

# BERT Model for classification
from transformers import AutoModelForSequenceClassification

# Optimizer for BERT
from torch.optim import AdamW

# Loss function for clasification 
from torch.nn import CrossEntropyLoss

In [2]:
# Check for GPU
print(torch.cuda.is_available())

True


In [3]:
# store the gpu
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using Device :  {device}")  

Using Device :  cuda


In [4]:
# Load the data

df = pd.read_csv("IMDB.csv")
print(df.head())

                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive


In [5]:
df.shape

(50000, 2)

In [6]:
df['sentiment'].value_counts()

sentiment
positive    25000
negative    25000
Name: count, dtype: int64

In [7]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [8]:
# Label Encoding
df['label'] = df['sentiment'].map({'positive':1, 'negative':0})
print(df.head())

                                              review sentiment  label
0  One of the other reviewers has mentioned that ...  positive      1
1  A wonderful little production. <br /><br />The...  positive      1
2  I thought this was a wonderful way to spend ti...  positive      1
3  Basically there's a family where a little boy ...  negative      0
4  Petter Mattei's "Love in the Time of Money" is...  positive      1


In [9]:
# Messy Inputs
df['review'][0]

"One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.<br /><br />I would say the main appeal of the show is due to the fa

In [10]:
# data proprocessing : Cleaning the text

def clean_text(text):
    
    # Remove html tags
    text = re.sub(r'<.*?>', "", text)   # .*? : anything - characters, numbers, symbols
    
    # Remove numbers
    text = re.sub(r'\d+', "", text)
    
    # Lower case
    text = text.lower()
    
    return text

In [11]:
# Clean dataset text
df['cleaned_review'] = df['review'].apply(clean_text)
print(df['cleaned_review'][0])

one of the other reviewers has mentioned that after watching just  oz episode you'll be hooked. they are right, as this is exactly what happened with me.the first thing that struck me about oz was its brutality and unflinching scenes of violence, which set in right from the word go. trust me, this is not a show for the faint hearted or timid. this show pulls no punches with regards to drugs, sex or violence. its is hardcore, in the classic use of the word.it is called oz as that is the nickname given to the oswald maximum security state penitentary. it focuses mainly on emerald city, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. em city is home to many..aryans, muslims, gangstas, latinos, christians, italians, irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.i would say the main appeal of the show is due to the fact that it goes where other shows woul

In [12]:
print("BEFORE:\n", df['review'][0])
print("\nAFTER:\n", df['cleaned_review'][0])


BEFORE:
 One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.<br /><br />I would say the main appeal of the show is due t

In [13]:
# Split the data

train_texts, temp_texts, train_labels, temp_labels = train_test_split(
    df['cleaned_review'].values,
    df['label'].values,
    test_size=0.2,
    random_state=42,
    stratify=df['label'].values
)

In [14]:
# Split further into validation and test data

val_texts, test_texts, val_labels, test_labels = train_test_split(
    temp_texts,
    temp_labels,
    test_size=0.5,
    random_state=42,
    stratify=temp_labels
)

In [15]:
print(f"Train size: {len(train_texts)}")
print(f"Validation size: {len(val_texts)}")
print(f"Test size: {len(test_texts)}")

Train size: 40000
Validation size: 5000
Test size: 5000


In [16]:
print(f"Train label distribution: {np.unique(train_labels, return_counts=True)}")
print(f"Val label distribution: {np.unique(val_labels, return_counts=True)}")
print(f"Test label distribution: {np.unique(test_labels, return_counts=True)}")

Train label distribution: (array([0, 1]), array([20000, 20000]))
Val label distribution: (array([0, 1]), array([2500, 2500]))
Test label distribution: (array([0, 1]), array([2500, 2500]))


In [17]:
# Tokenization
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')  # Bert's Tokenizer

In [18]:
sample_text = "I love this man prem !!"
tokens = tokenizer(sample_text)
print(tokens)

{'input_ids': [101, 1045, 2293, 2023, 2158, 26563, 999, 999, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1]}


In [19]:
print(tokenizer.convert_ids_to_tokens(tokens['input_ids']))

['[CLS]', 'i', 'love', 'this', 'man', 'prem', '!', '!', '[SEP]']


In [ ]:
sample_texts = ["I loved this movie !!", "Bad."]
tokens = tokenizer(
    sample_texts,
    padding = True,
    truncation = True,
    max_length=20,
    return_tensors = 'pt'
)

print("input_ids:\n", tokens['input_ids'])
print("\nattention_mask:\n", tokens['attention_mask'])

# This showa the example of padding. where sentence Bad. is short so to satisfy the max length criteria the blank spaces are filled by 0s
# So in attention mask this padding are valued as 0 so BERT shouldn't waste its time finding meaning in this padding value

input_ids:
 tensor([[ 101, 1045, 3866, 2023, 3185,  999,  999,  102],
        [ 101, 2919, 1012,  102,    0,    0,    0,    0]])

attention_mask:
 tensor([[1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 0, 0, 0, 0]])


Tokenizer : It converts raw text into three arrays that BERT can understand — input_ids, attention_mask and token_type_ids.

In [ ]:
# For choosing max length of tokens, need to analyze avg and max of entire dataset
lengths = df['cleaned_review'].apply(lambda x: len(tokenizer.tokenize(x)))
print(f"Average length: {lengths.mean()}")
print(f"Max length: {lengths.max()}")
print(f"90th percentile: {lengths.quantile(0.90)}")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (529 > 512). Running this sequence through the model will result in indexing errors


Average length: 290.72204
Max length: 3038
90th percentile: 569.0


In [23]:
# 90th percentile means — 90% of our reviews are under 569 tokens. So if we set max_length=256, 
# we're covering the majority of reviews reasonably well. The very long ones just get cut off a little.

MAX_LENGTH = 256

train_encodings = tokenizer(
    list(train_texts),
    padding = True,
    truncation = True,
    max_length = MAX_LENGTH,
    return_tensors = 'pt'
)

In [ ]:
# Tokenizing Validation Set

val_encodings = tokenizer(
    list(val_texts),
    padding = True,
    truncation = True,
    max_length = MAX_LENGTH,
    return_tensors = 'pt'
)

In [ ]:
# Tokenizing Test Set

test_encodings = tokenizer(
    list(test_texts),
    padding = True,
    truncation = True,
    max_length = MAX_LENGTH,
    return_tensors = 'pt'
)

In [26]:
# validate tokenization

print(f"Train input_ids shape : {train_encodings['input_ids'].shape}")
print(f"Validation input_ids shape : {val_encodings['input_ids'].shape}")
print(f"Test input_ids shape : {test_encodings['input_ids'].shape}")

Train input_ids shape : torch.Size([40000, 256])
Validation input_ids shape : torch.Size([5000, 256])
Test input_ids shape : torch.Size([5000, 256])


In [32]:
# Encodings are in seperate variables - Patch them together in specific format using Pytorch so model can be feeded

class IMDBDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
        
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        item = {key : val[idx] for key, val in self.encodings.items()}  # Dictionary Comprehension
        item['labels'] = torch.tensor(self.labels[idx])
        # Stores the desried index data in dictionary item and returns it 
        return item
    
        # So when PyTorch asks for sample 5, it gets back a neat dictionary:
        # {
        #   'input_ids': tensor of 256 numbers,
        #   'attention_mask': tensor of 256 numbers,
        #   'labels': tensor(1)  # positive
        # }

In [33]:
# Create Three dataset object

train_dataset = IMDBDataset(train_encodings, train_labels)
val_dataset = IMDBDataset(val_encodings, val_labels)
test_dataset = IMDBDataset(test_encodings, test_labels)

In [34]:
print(f"Train dataset size: {len(train_dataset)}")
print(f"Val dataset size: {len(val_dataset)}")
print(f"Test dataset size: {len(test_dataset)}")

Train dataset size: 40000
Val dataset size: 5000
Test dataset size: 5000


In [36]:
# Dataloader for batches 

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

In [38]:
# Load actual Model

model = AutoModelForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels = 2  # Our classification has 2 output +ve or -ve
)

d:\Prem\Codes\Innomatics Internship\innomatics-genai-internship-2026\IN226105802_NLP\venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\premv\.cache\huggingface\hub\models--bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1571.5

In [39]:
print(model)

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [42]:
# Move Model to GPU

model = model.to(device)
print(f"Model is on {next(model.parameters()).device}")

Model is on cuda:0


In [45]:
# Optimizer for BERT
optimizer = AdamW(model.parameters(), lr=2e-5)

In [47]:
# Loss function for classification
loss_fn = CrossEntropyLoss()

In [ ]:
# One full pass through all 40,000 training samples = one epoch.

# Traning Loop for BERT

def train_epoch(model, dataloader, optimizer, device):
    # Set to traning mode
    model.train()
    total_loss = 0
    
    for batch in dataloader:
        # Zero out the gradient
        optimizer.zero_grad()
        
        # Move batch to GPU
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        # Forward pass - Feed data to BERT and get predictions
        output = model(
            input_ids= input_ids,
            attention_mask = attention_mask,
            labels = labels     # as we pass labels directly into model, loss is calculated internally, so here is no use of loss_fn we created earlier
        )
        
        # back propogation
        output.loss.backword()
        
        # update weights
        output.step()
        